In [1]:
import duckdb

In [2]:
con = duckdb.connect()

In [3]:
con.execute("""
    CREATE OR REPLACE VIEW parquet_scan AS
    SELECT *
    FROM read_parquet('../data/parquet/2nd-slice/**/*.parquet')
""")

schema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'parquet_scan'
""").fetchdf()

schema

,column_name,data_type
0,vendor_id,UTINYINT
1,pickup_datetime,TIMESTAMP_NS
2,dropoff_datetime,TIMESTAMP_NS
3,passenger_count,UTINYINT
4,trip_distance,FLOAT
5,rate_code_id,UTINYINT
6,store_and_forward_flag,VARCHAR
7,pickup_location_id,SMALLINT
8,dropoff_location_id,SMALLINT
9,payment_type_id,UTINYINT


In [13]:
con.execute("""
COPY (
    SELECT
        CAST(vendor_id AS TINYINT) AS vendor_id,
        CAST(pickup_datetime AS TIMESTAMP) AS pickup_datetime,
        CAST(dropoff_datetime AS TIMESTAMP) AS dropoff_datetime,
        CAST(passenger_count AS SMALLINT) AS passenger_count,
        CAST(trip_distance AS DOUBLE) AS trip_distance,
        CAST(rate_code_id AS TINYINT) AS rate_code_id,
        CAST(store_and_forward_flag AS VARCHAR) AS store_and_forward_flag,
        CAST(pickup_location_id AS SMALLINT) AS pickup_location_id,
        CAST(dropoff_location_id AS SMALLINT) AS dropoff_location_id,
        CAST(payment_type_id AS TINYINT) AS payment_type_id,
        CAST(fare_amount AS DOUBLE) AS fare_amount,
        CAST(extra_charge AS DOUBLE) AS extra_charge,
        CAST(metered_rate_tax AS DOUBLE) AS metered_rate_tax,
        CAST(tip_amount AS DOUBLE) AS tip_amount,
        CAST(tolls_amount AS DOUBLE) AS tolls_amount,
        CAST(improvement_surcharge AS DOUBLE) AS improvement_surcharge,
        CAST(total_amount AS DOUBLE) AS total_amount,
        CAST(congestion_surcharge AS DOUBLE) AS congestion_surcharge,
        CAST(airport_fee AS DOUBLE) AS airport_fee,
        CAST(cbd_congestion_fee AS DOUBLE) AS cbd_congestion_fee,
        CAST(currency AS VARCHAR) AS currency,
        CAST(trip_distance_unit AS VARCHAR) AS trip_distance_unit,
        CAST(suspicious AS BOOLEAN) AS suspicious,
        CAST(year AS INTEGER) AS year,
        CAST(month AS INTEGER) AS month
    FROM read_parquet('../data/parquet/2nd-slice_compacted/**/*.parquet')
)
TO '../data/parquet/2nd-slice_spark_compatible'
(FORMAT PARQUET, PARTITION_BY (year, month));
""")

In [6]:
files = con.execute("""
    SELECT file
    FROM glob('../data/parquet/2nd-slice/**/*.parquet')
""").fetchall()

In [7]:
import os

for (file_path,) in files:
    df = con.execute(f"""
            SELECT
                CAST(vendor_id AS TINYINT) AS vendor_id,
                CAST(pickup_datetime AS TIMESTAMP) AS pickup_datetime,
                CAST(dropoff_datetime AS TIMESTAMP) AS dropoff_datetime,
                CAST(passenger_count AS SMALLINT) AS passenger_count,
                CAST(trip_distance AS DOUBLE) AS trip_distance,
                CAST(rate_code_id AS TINYINT) AS rate_code_id,
                CAST(store_and_forward_flag AS VARCHAR) AS store_and_forward_flag,
                CAST(pickup_location_id AS SMALLINT) AS pickup_location_id,
                CAST(dropoff_location_id AS SMALLINT) AS dropoff_location_id,
                CAST(payment_type_id AS TINYINT) AS payment_type_id,
                CAST(fare_amount AS DOUBLE) AS fare_amount,
                CAST(extra_charge AS DOUBLE) AS extra_charge,
                CAST(metered_rate_tax AS DOUBLE) AS metered_rate_tax,
                CAST(tip_amount AS DOUBLE) AS tip_amount,
                CAST(tolls_amount AS DOUBLE) AS tolls_amount,
                CAST(improvement_surcharge AS DOUBLE) AS improvement_surcharge,
                CAST(total_amount AS DOUBLE) AS total_amount,
                CAST(congestion_surcharge AS DOUBLE) AS congestion_surcharge,
                CAST(airport_fee AS DOUBLE) AS airport_fee,
                CAST(cbd_congestion_fee AS DOUBLE) AS cbd_congestion_fee,
                CAST(currency AS VARCHAR) AS currency,
                CAST(trip_distance_unit AS VARCHAR) AS trip_distance_unit,
                CAST(suspicious AS BOOLEAN) AS suspicious,
                CAST(year AS INTEGER) AS year,
                CAST(month AS INTEGER) AS month
            FROM read_parquet('{file_path}')
        """).df()

    # Build output path
    year = df['year'][0]
    month = df['month'][0]
    out_dir = f"../data/parquet/2nd-slice_spark_compatible/year={year}/month={month}"
    os.makedirs(out_dir, exist_ok=True)

    out_file = os.path.join(out_dir, os.path.basename(file_path))

    df.to_parquet(out_file)
